# Crop Rules — Data Extraction
Reads `crop_rules.json` directly. Raw string keys — no ID mapping.

In [19]:
import json
import re
import os
import pandas as pd

JSON_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         '..', '..', 'Data_Setup', 'Datasets', 'crop_rules.json')

with open(JSON_PATH, 'r') as f:
    rules = json.load(f)

def clean_key(key):
    """'>70' → 'above_70', '<40' → 'below_40', '15-20' → '15_20'."""
    key = re.sub(r'^>', 'above_', str(key))
    key = re.sub(r'^<', 'below_', key)
    return key.replace('-', '_').lower()

def build_wide(section_key, prefix):
    """One row per crop with prefixed score columns."""
    rows = []
    for crop, scores in rules[section_key].items():
        if crop.startswith('_'):
            continue
        row = {'crop': crop}
        row.update({f"{prefix}{clean_key(k)}": v for k, v in scores.items()})
        rows.append(row)
    return pd.DataFrame(rows)

print('Sections loaded:', [k for k in rules if not k.startswith('_')])

Sections loaded: ['CROP_RULES', 'crop_season_score', 'soil_compatibility', 'water_compatibility', 'temperature_compatibility', 'humidity_compatibility', 'soil_moisture_compatibility', 'sunlight_compatibility', 'irrigation_frequency_compatibility', 'ph_compatibility', 'soil_adjustment_rules', 'season_adjustment_rules', 'temperature_adjustment_rules', 'humidity_moisture_rules', 'ph_adjustment_rules']


## 1. Crop–Soil Compatibility
Composite key: `crop + soil_type`

In [20]:
rows = []
for crop, scores in rules['soil_compatibility'].items():
    if crop.startswith('_'):
        continue
    for soil_type, score in scores.items():
        rows.append({'crop': crop, 'soil_type': soil_type, 'score': score})

crop_soil_df = pd.DataFrame(rows)
print('Shape:', crop_soil_df.shape)
crop_soil_df

Shape: (9, 3)


,crop,soil_type,score
0,Rice,Sandy,0.20
1,Rice,Loamy,0.70
2,Rice,Clay,0.95
3,Maize,Sandy,0.50
4,Maize,Loamy,0.95
5,Maize,Clay,0.30
6,Cotton,Sandy,0.50
7,Cotton,Loamy,0.90
8,Cotton,Clay,0.20


## 2. Crop–Season Compatibility
Composite key: `crop + season`

In [21]:
rows = []
for crop, scores in rules['crop_season_score'].items():
    if crop.startswith('_'):
        continue
    for season, score in scores.items():
        rows.append({'crop': crop, 'season': season, 'score': score})

crop_season_df = pd.DataFrame(rows)
print('Shape:', crop_season_df.shape)
crop_season_df

Shape: (9, 3)


,crop,season,score
0,Rice,Rainy,0.95
1,Rice,Winter,0.20
2,Rice,Summer,0.40
3,Maize,Rainy,0.85
4,Maize,Winter,0.30
5,Maize,Summer,0.60
6,Cotton,Rainy,0.80
7,Cotton,Winter,0.15
8,Cotton,Summer,0.50


## 3. Water Source Compatibility
Composite key: `crop + water_source` *(River, Groundwater, Recycled only)*

In [22]:
WATER_SOURCES = ['River', 'Groundwater', 'Recycled']

rows = []
for crop, scores in rules['water_compatibility'].items():
    if crop.startswith('_'):
        continue
    for src in WATER_SOURCES:
        rows.append({'crop': crop, 'water_source': src, 'score': scores[src]})

water_source_df = pd.DataFrame(rows)
print('Shape:', water_source_df.shape)
water_source_df

Shape: (9, 3)


,crop,water_source,score
0,Rice,River,0.95
1,Rice,Groundwater,0.75
2,Rice,Recycled,0.40
3,Maize,River,0.70
4,Maize,Groundwater,0.90
5,Maize,Recycled,0.55
6,Cotton,River,0.65
7,Cotton,Groundwater,0.90
8,Cotton,Recycled,0.50


## 4. Climate Compatibility
Unique key: `crop`
Columns: `temp_*`, `humidity_*`, `sunlight_*`

In [23]:
climate_compatibility_df = (
    build_wide('temperature_compatibility', 'temp_')
    .merge(build_wide('humidity_compatibility',  'humidity_'),  on='crop')
    .merge(build_wide('sunlight_compatibility',  'sunlight_'),  on='crop')
)

print('Shape:', climate_compatibility_df.shape)
climate_compatibility_df

Shape: (3, 10)


,crop,temp_15_20,temp_20_30,temp_30_40,humidity_below_40,humidity_40_70,humidity_above_70,sunlight_below_4,sunlight_4_8,sunlight_above_8
0,Rice,0.3,0.80,0.5,0.2,0.75,0.95,0.4,0.80,0.80
1,Maize,0.4,0.85,0.7,0.5,0.80,0.50,0.4,0.85,0.95
2,Cotton,0.2,0.75,0.9,0.7,0.70,0.40,0.3,0.75,0.90


## 5. Soil–Climate Compatibility
Unique key: `crop_id`  
Columns: `moisture_below_30`, `moisture_30_60`, `moisture_above_60`, `ph_acidic`, `ph_neutral`, `ph_alkaline`

In [24]:
soil_climate_df = build_wide('soil_moisture_compatibility', 'moisture_').merge(
    build_wide('ph_compatibility', 'ph_'),
    on='crop'
)

print('Shape:', soil_climate_df.shape)
soil_climate_df

Shape: (3, 7)


,crop,moisture_below_30,moisture_30_60,moisture_above_60,ph_acidic,ph_neutral,ph_alkaline
0,Rice,0.2,0.75,0.95,0.80,0.85,0.60
1,Maize,0.5,0.90,0.40,0.60,0.95,0.60
2,Cotton,0.5,0.85,0.30,0.55,0.90,0.75


## 6. Irrigation Frequency Compatibility
Unique key: `crop`
Columns: `irr_freq_below_2`, `irr_freq_3_4`, `irr_freq_above_5`

In [25]:
irrigation_frequency_df = build_wide('irrigation_frequency_compatibility', 'irr_freq_')

print('Shape:', irrigation_frequency_df.shape)
irrigation_frequency_df

Shape: (3, 5)


,crop,irr_freq_below_2,irr_freq_3_4,irr_freq_above_5,irr_freq_1_2
0,Rice,0.2,0.7,0.90,NaN
1,Maize,NaN,0.9,0.85,0.4
2,Cotton,NaN,0.7,0.60,0.5


---
## Summary

| DataFrame | Key | Rows |
|---|---|---|
| `crop_soil_df` | crop + soil_type | 9 |
| `crop_season_df` | crop + season | 9 |
| `water_source_df` | crop + water_source | 9 |
| `climate_compatibility_df` | crop | 3 |
| `soil_climate_df` | crop | 3 |